# Champions League Match Intelligence — exploring the curated layer

**Who this is for:** anyone on the team who wants to look at the data product without
reading the pipeline code first. It queries the curated tables only — the same tables the
Streamlit app uses — so whatever you see here is what a consumer of our data sees.

**Before you start**

```bash
make up          # PostgreSQL
make init        # schemas
make run --from-samples   # or: make ingest && make transform && make dq
```

Then run this notebook with the project venv as its kernel:

```bash
make notebook    # starts Jupyter with the right kernel
```

**A rule worth keeping:** exploration belongs here, pipeline logic belongs in `src/deng/`
and `sql/`. If a query here turns out to be useful for the data product, move it into a
transformation and add a test — a notebook is not reproducible by the pipeline.


## 1. Connect


In [ ]:
import os
import sys

import pandas as pd
import psycopg

sys.path.insert(0, "../src")  # works without installing the package
from deng.config import get_settings  # noqa: E402

os.chdir("..")  # settings read .env from the repo root
settings = get_settings()
conn = psycopg.connect(settings.postgres_dsn)


def q(sql, **params):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params or None)


q("SELECT current_database() AS db, now()::date AS today")

## 2. What is in the warehouse?

Every table states its grain in the database itself, so this is not documentation that can
drift — it is `COMMENT ON TABLE`, read back live.


In [ ]:
q("""
SELECT c.table_schema || '.' || c.table_name AS table_name,
       obj_description((c.table_schema || '.' || c.table_name)::regclass) AS grain
  FROM information_schema.tables c
 WHERE c.table_schema IN ('raw','staging','curated','meta')
   AND c.table_type = 'BASE TABLE'
 ORDER BY 1
""")

In [ ]:
q("""
SELECT 'raw.football_data' AS t, count(*) FROM raw.football_data
UNION ALL SELECT 'staging.matches', count(*) FROM staging.matches
UNION ALL SELECT 'curated.dim_team', count(*) FROM curated.dim_team
UNION ALL SELECT 'curated.fact_match', count(*) FROM curated.fact_match
UNION ALL SELECT 'curated.fact_team_match_form', count(*) FROM curated.fact_team_match_form
UNION ALL SELECT 'meta.pipeline_runs', count(*) FROM meta.pipeline_runs
UNION ALL SELECT 'meta.dq_results', count(*) FROM meta.dq_results
""")

## 3. The matches

`fact_match` — one row per Champions League match. Note `is_upcoming`: it is derived from
the kick-off time, **not** from `status`. Querying the API with `?status=SCHEDULED` returns
rows whose stored status is `TIMED`, so a filter on the status string silently returns
nothing (see `docs/evidence/api-exploration.md` §7).


In [ ]:
q("""
SELECT status, is_finished, is_upcoming, count(*) AS matches,
       min(match_date) AS first_date, max(match_date) AS last_date
  FROM curated.fact_match
 GROUP BY 1,2,3 ORDER BY 1
""")

In [ ]:
# The next matches, the way the app shows them
q("""
SELECT m.utc_kickoff, m.matchday,
       h.short_name AS home, a.short_name AS away, h.venue
  FROM curated.fact_match m
  JOIN curated.dim_team h ON h.team_id = m.home_team_id
  JOIN curated.dim_team a ON a.team_id = m.away_team_id
 WHERE m.is_upcoming
 ORDER BY m.utc_kickoff
 LIMIT 10
""")

## 4. Results and the target label

`outcome` is computed from the goals rather than copied from the API's `score.winner`, so
the label can never disagree with the numbers in the same row. It is the target variable
for any later model.


In [ ]:
q("""
SELECT outcome, count(*) AS matches,
       round(100.0*count(*)/sum(count(*)) OVER (), 1) AS pct
  FROM curated.fact_match WHERE is_finished
 GROUP BY 1 ORDER BY 2 DESC
""")

In [ ]:
q("""
SELECT h.short_name AS home, m.home_goals, m.away_goals, a.short_name AS away,
       m.outcome, m.match_date
  FROM curated.fact_match m
  JOIN curated.dim_team h ON h.team_id = m.home_team_id
  JOIN curated.dim_team a ON a.team_id = m.away_team_id
 WHERE m.is_finished ORDER BY m.utc_kickoff DESC LIMIT 10
""")

## 5. Form — and why `matches_considered` matters

`fact_team_match_form` holds one row per team per match: that team's form **going into**
the match, built only from games finished before kick-off.

Two things to be careful with:

1. `matches_considered` says how many games the window rests on. Early in the season it is
   0 or 1. Comparing a 5-match form with a 1-match form without saying so is misleading.
2. For 11 of the 36 clubs the free API tier carries no domestic league
   (`has_domestic_coverage = false`), so their window stays thin for longer.


In [ ]:
q("""
SELECT matches_considered, count(*) AS rows
  FROM curated.fact_team_match_form GROUP BY 1 ORDER BY 1
""")

In [ ]:
q("""
SELECT d.short_name, d.country, d.competitions, d.has_domestic_coverage
  FROM curated.dim_team d
 WHERE NOT d.has_domestic_coverage ORDER BY d.short_name
""")

In [ ]:
# Form of both teams for one upcoming match - the core of the app's page
q("""
SELECT h.short_name AS home, a.short_name AS away, m.utc_kickoff,
       fh.matches_considered AS home_n, fh.points_last_5 AS home_pts,
       fh.goals_scored_last_5 AS home_gf, fh.goals_conceded_last_5 AS home_ga,
       fa.matches_considered AS away_n, fa.points_last_5 AS away_pts,
       fa.goals_scored_last_5 AS away_gf, fa.goals_conceded_last_5 AS away_ga
  FROM curated.fact_match m
  JOIN curated.dim_team h ON h.team_id = m.home_team_id
  JOIN curated.dim_team a ON a.team_id = m.away_team_id
  JOIN curated.fact_team_match_form fh ON fh.match_id = m.match_id AND fh.team_id = m.home_team_id
  JOIN curated.fact_team_match_form fa ON fa.match_id = m.match_id AND fa.team_id = m.away_team_id
 WHERE m.is_upcoming ORDER BY m.utc_kickoff LIMIT 5
""")

## 6. Verify the leakage guard yourself

Do not take the pipeline's word for it. This query recomputes, for every form row, how many
matches the team had actually completed before kick-off, and compares it with what we
stored. Anything other than an empty result is a bug worth reporting.


In [ ]:
q("""
SELECT f.match_id, f.team_id, f.matches_considered AS stored,
       (SELECT count(*) FROM curated.fact_match p
         WHERE p.is_finished AND p.utc_kickoff < m.utc_kickoff
           AND (p.home_team_id = f.team_id OR p.away_team_id = f.team_id)) AS actually_available
  FROM curated.fact_team_match_form f
  JOIN curated.fact_match m USING (match_id)
 WHERE f.matches_considered > (SELECT count(*) FROM curated.fact_match p
         WHERE p.is_finished AND p.utc_kickoff < m.utc_kickoff
           AND (p.home_team_id = f.team_id OR p.away_team_id = f.team_id))
""")

## 7. Pipeline health

Runs and data-quality results live in the database, so "did yesterday's run work?" is a
query rather than an archaeology exercise in container logs.


In [ ]:
q("""
SELECT pipeline_name, logical_date, status, rows_extracted, rows_loaded, rows_updated,
       round(extract(epoch FROM finished_at - started_at)::numeric, 2) AS seconds
  FROM meta.pipeline_runs ORDER BY started_at DESC LIMIT 10
""")

In [ ]:
q("""
SELECT check_name, severity, passed, observed, checked_at
  FROM meta.dq_results
 WHERE checked_at = (SELECT max(checked_at) FROM meta.dq_results)
 ORDER BY passed, check_name
""")

## 8. Your turn

Ideas that would genuinely help the project, roughly in order of usefulness:

- **Home advantage:** is the home win rate different from the away win rate, and by how
  much? It is the baseline any model has to beat.
- **Does form predict anything?** Join `fact_team_match_form` to finished matches and check
  whether `points_last_5` correlates with the outcome. Only use rows with
  `matches_considered >= 3`, and say so.
- **Standings over time:** `staging.standings` keeps one snapshot per ingestion date. After
  a few days of runs you can plot a team's position as a time series.
- **What is missing?** Compare `staging.matches` with `curated.fact_match` and find out
  whether anything gets dropped, and why.

If a query becomes part of the product, move it into `sql/transform/` with a test — this
notebook is for looking, not for producing.


In [ ]:
conn.close()